In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
#plt.style.use('seaborn-darkgrid')
import requests
import os, glob, zipfile, regionmask, cdsapi, pickle
from pathlib import Path
import xarray as xr

C:\Users\nb\anaconda3\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


# MAKING CLIMATE GEO-DATA

# The Coffee Growing Regions and ShapeFile

In [ ]:
def make_climate_csv(pname,fname,coffee_geo,date_pos=3):
    outdict={}
    ds=xr.open_dataset(pname+fname)
    date_index=fname.split('_')[date_pos]
    state_mask = regionmask.from_geopandas(coffee_geo).mask(ds.lon, ds.lat)
    var_name=list(ds.variables)[0]
    values_list=ds.groupby(state_mask).mean()[var_name].values.reshape(1,-1)
    outdict[date_index]=list(values_list[0])
    return pd.DataFrame(outdict).T

In [ ]:
def make_climate_xr(ds):
    coffee_geo=gpd.read_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",epsg=4326)
    temp_xr = ds.sel(lat=slice(coffee_geo.total_bounds[3], coffee_geo.total_bounds[1]),
                            lon=slice(coffee_geo.total_bounds[0], coffee_geo.total_bounds[2]))
    
    coffee_mask = regionmask.from_geopandas(coffee_geo).mask(temp_xr.lon, temp_xr.lat)
    coffee_xr = ds.where(~np.isnan(coffee_mask))

    return coffee_xr

In [ ]:
variables=["vapour_pressure_deficit_at_maximum_temperature", "precipitation_flux","solar_radiation_flux","reference_evapotranspiration"]
def get_copernicus_data(area,years=[1990+i for i in range(36)],
                        months=["01", "02", "03","04", "05", "06","07", "08", "09","10", "11", "12"],
                        days=["01", "02", "03",
                              "04", "05", "06",
                              "07", "08", "09",
                              "10", "11", "12",
                              "13", "14", "15",
                              "16", "17", "18",
                              "19", "20", "21",
                              "22", "23", "24",
                              "25", "26", "27",
                              "28", "29", "30","31"],
                        var_name=variables[0]):
    
    
    coords_dict={'brazil':[-5, -75, -35, -35],
                 'colombia':[15, -80, -5, -65]}
    
    dataset = "sis-agrometeorological-indicators"
    for year in years:
        request = {
            "variable": var_name,
            "year": [year],
            "month": months,
            "day": days,
            "version": "2_0",
            "area": coords_dict[area]
        }

        client = cdsapi.Client()
        var_name=var_name.replace("flux","flex")
        save_path={'brazil':f"C://Users/nb/Desktop/brazil_climate_data/{var_name}_{year}.zip",
                   'columbia':f"C://Users/nb/Desktop/colombia_climate_data/{var_name}_{year}.zip"}
        client.retrieve(dataset, request).download(save_path[area])

## BRAZIL

In [12]:
estados=['BA','MG','ES','SP','PR']

In [ ]:
geojson_file=gpd.read_file("C://Users/nb/Desktop/bra_adm1.geojson")
geojson_file.columns=['estado','est_code','shape_id','brazil','shape_type','geometry']
coffee_estados =['Minas Gerais', 'Bahia', 'Espirito Santo', 'Sao Paulo', 'Parana']
coffee_geo=geojson_file[geojson_file['estado'].isin(coffee_estados)]
coffee_geo=coffee_geo.reset_index(drop=True)
coffee_geo=coffee_geo[['estado','est_code','geometry']]
coffee_geo.to_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",index=False)

In [ ]:
coffee_geo=gpd.read_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",epsg=4326)

In [ ]:
#for var in variables[1:]:
get_copernicus_data(area='brazil',months=["12"],years=[1989],var_name= "precipitation_flux")
get_copernicus_data(area='brazil',months=["01"],years=[2026],var_name= "precipitation_flux")

In [ ]:
raw_files=['precipitation_flux_raw',
           'solar_radiation_flux_raw',
           'vapour_pressure_deficit_at_max_temp_raw',
           'reference_evapotranspiration_raw'
            ]

for raw in raw_files:
    file_txt=raw.replace('_raw','')
    raw_zip_folder = f'C://Users/nb/Desktop/brazil_climate_data/{raw}/'
    extract_folder = f'C://Users/nb/Desktop/brazil_climate_data/{file_txt}/'
    
    #output_file = f'combined_{file_txt}_1990_1999.nc'

    os.makedirs(raw_zip_folder, exist_ok=True)

    print("Extracting files...")
    zip_files = glob.glob(os.path.join(raw_zip_folder, "*.zip"))

    for zip_path in zip_files:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            nc_files = [f for f in zip_ref.namelist() if f.endswith('.nc')]
            zip_ref.extractall(path=extract_folder, members=nc_files)
        print(f"Done: {os.path.basename(zip_path)}")


In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/precipitation_flex/"
prec_flex_xr = xr.open_mfdataset(pname+'*.nc',preprocess=make_climate_xr)

In [ ]:
prec_flex_xr.to_netcdf("C://Users/nb/Desktop/brazil_climate_data/coffee_prec_flex.nc")

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/solar_radiation_flex/"
rad_flex_xr = xr.open_mfdataset(pname+'*.nc',preprocess=make_climate_xr)

In [ ]:
rad_flex_xr.to_netcdf("C://Users/nb/Desktop/brazil_climate_data/coffee_rad_flex.nc")

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/reference_evapotranspiration/"
pet_xr = xr.open_mfdataset(pname+'*.nc',preprocess=make_climate_xr)

In [ ]:
pet_xr.to_netcdf("C://Users/nb/Desktop/brazil_climate_data/coffee_pet.nc")

### vapour pressure deficit at maximum temperature

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/vapour_pressure_deficit_at_max_temp/"
vpd_max_xr =xr.open_mfdataset(pname+'*.nc',preprocess=make_climate_xr)

In [ ]:
vpd_max_xr.to_netcdf("C://Users/nb/Desktop/brazil_climate_data/coffee_vpd_max.nc")

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/vapour_pressure_deficit_at_max_temp_cn/"
vpd_max_xr =xr.open_mfdataset(pname+'*.nc',preprocess=make_climate_xr)

In [ ]:
vpd_max_xr.to_netcdf("C://Users/nb/Desktop/brazil_climate_data/coffee_vpd_max_cn.nc")

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/vapour_pressure_deficit_at_max_temp//"
fnames=os.listdir(pname)
coffee_geo=gpd.read_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",epsg=4326) ##orignally also 4326
vpdmax_df=[]
for fname in fnames:
    test=make_climate_csv(pname,fname,coffee_geo=coffee_geo)
    vpdmax_df.append(test)
vpdmax_df_main=pd.concat(vpdmax_df,axis=0)
vpdmax_df_main=vpdmax_df_main.reset_index()
vpdmax_df_main.columns=['date']+estados
#vpdmax_df_main.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_vpdmax_coffeevJune.csv",index=False)

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/vapour_pressure_deficit_at_max_temp_cn//"
fnames=os.listdir(pname)
coffee_geo=gpd.read_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",epsg=4326) ##orignally also 4326
vpdmax_df=[]
for fname in fnames:
    test=make_climate_csv(pname,fname,coffee_geo=coffee_geo)
    vpdmax_df.append(test)
vpdmax_df_main=pd.concat(vpdmax_df,axis=0)
vpdmax_df_main=vpdmax_df_main.reset_index()
vpdmax_df_main.columns=['date']+estados
#vpdmax_df_main.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_vpdmax_coffeevJune_cn.csv",index=False)

In [ ]:
p1=pd.read_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_vpdmax_coffeevJune.csv")
print(p1.shape)
print(vpdmax_df_main.shape)
vpdmax_df_main['date']=vpdmax_df_main['date'].astype(int)
p2=pd.concat([p1,vpdmax_df_main],axis=0)
p2=p2[(p2['date']>=19891229)&(p2['date']<=20260104)]
p2=p2.sort_values(by='date').reset_index(drop=True)

In [ ]:
p2.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_vpdmax_coffeevJune_cn.csv",index=False)

### precipitation flux

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/precipitation_flex/"
fnames=os.listdir(pname)
coffee_geo=gpd.read_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",epsg=4326) ##orignally also 4326
prec_flexdf=[]
for fname in fnames:
    test=make_climate_csv(pname,fname,coffee_geo=coffee_geo)
    prec_flexdf.append(test)
prec_flexdf_main=pd.concat(prec_flexdf,axis=0)
prec_flexdf_main=prec_flexdf_main.reset_index()
prec_flexdf_main.columns=['date']+estados
prec_flexdf_main.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_prec_flex_coffeevJune.csv",index=False)

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/precipitation_flex_cn/"
fnames=os.listdir(pname)
coffee_geo=gpd.read_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",epsg=4326) ##orignally also 4326
prec_flexdf=[]
for fname in fnames:
    test=make_climate_csv(pname,fname,coffee_geo=coffee_geo)
    prec_flexdf.append(test)
prec_flexdf_main=pd.concat(prec_flexdf,axis=0)
prec_flexdf_main=prec_flexdf_main.reset_index()
prec_flexdf_main.columns=['date']+estados
#prec_flexdf_main.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_prec_flex_coffeevJune.csv",index=False)

In [ ]:
p1=pd.read_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_prec_flex_coffeevJune.csv")
print(p1.shape)
print(prec_flexdf_main.shape)
prec_flexdf_main['date']=prec_flexdf_main['date'].astype(int)
p2=pd.concat([p1,prec_flexdf_main],axis=0)
p2=p2[(p2['date']>=19891229)&(p2['date']<=20260104)]
p2=p2.sort_values(by='date').reset_index(drop=True)

In [ ]:
p2.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_prec_flex_coffeevJune_cn.csv",index=False)

### solar radiation flux

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/solar_radiation_flex//"
fnames=os.listdir(pname)
coffee_geo=gpd.read_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",epsg=4326) ##orignally also 4326
rad_flexdf=[]
for fname in fnames:
    test=make_climate_csv(pname,fname,coffee_geo=coffee_geo)
    rad_flexdf.append(test)
rad_flexdf_main=pd.concat(rad_flexdf,axis=0)
rad_flexdf_main=rad_flexdf_main.reset_index()
rad_flexdf_main.columns=['date']+estados
rad_flexdf_main.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_rad_flex_coffeevJune.csv",index=False)

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/solar_radiation_flex_cn//"
fnames=os.listdir(pname)
coffee_geo=gpd.read_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",epsg=4326) ##orignally also 4326
rad_flexdf=[]
for fname in fnames:
    test=make_climate_csv(pname,fname,coffee_geo=coffee_geo)
    rad_flexdf.append(test)
rad_flexdf_main=pd.concat(rad_flexdf,axis=0)
rad_flexdf_main=rad_flexdf_main.reset_index()
rad_flexdf_main.columns=['date']+estados
#rad_flexdf_main.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_rad_flex_coffeevJune.csv",index=False)

In [ ]:
p1=pd.read_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_rad_flex_coffeevJune.csv")
print(p1.shape)
print(rad_flexdf_main.shape)
rad_flexdf_main['date']=rad_flexdf_main['date'].astype(int)
p2=pd.concat([p1,rad_flexdf_main],axis=0)
p2=p2[(p2['date']>=19891229)&(p2['date']<=20260104)]
p2=p2.sort_values(by='date').reset_index(drop=True)

In [ ]:
p2.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_rad_flex_coffeevJune_cn.csv",index=False)

### reference evapotranspiration

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/reference_evapotranspiration/"
fnames=os.listdir(pname)
coffee_geo=gpd.read_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",epsg=4326) ##orignally also 4326
petdf=[]
for fname in fnames:
    test=make_climate_csv(pname,fname,coffee_geo=coffee_geo)
    petdf.append(test)
petdf_main=pd.concat(petdf,axis=0)
petdf_main=petdf_main.reset_index()
petdf_main.columns=['date']+estados
petdf_main.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_pet_coffeevJune.csv",index=False)

In [ ]:
pname="C://Users/nb/Desktop/brazil_climate_data/reference_evapotranspiration_cn/"
fnames=os.listdir(pname)
coffee_geo=gpd.read_file("C://Users/nb/Desktop/brazil_climate_data/coffee_geo.shp",epsg=4326) ##orignally also 4326
petdf=[]
for fname in fnames:
    test=make_climate_csv(pname,fname,coffee_geo=coffee_geo)
    petdf.append(test)
petdf_main=pd.concat(petdf,axis=0)
petdf_main=petdf_main.reset_index()
petdf_main.columns=['date']+estados
#petdf_main.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_pet_coffeevJune.csv",index=False)

In [ ]:
p1=pd.read_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_pet_coffeevJune.csv")
print(p1.shape)
print(petdf_main.shape)
petdf_main['date']=petdf_main['date'].astype(int)
p2=pd.concat([p1,petdf_main],axis=0)
p2=p2[(p2['date']>=19891229)&(p2['date']<=20260104)]
p2=p2.sort_values(by='date').reset_index(drop=True)

In [ ]:
p2.to_csv("C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/daily_pet_coffeevJune_cn.csv",index=False)

# Z-Score Climate Anomalies

In [44]:
def save_dictionaries_as_pickle(in_dict,save_path,dict_name):
    
    pickle_file = open(save_path+dict_name, 'wb')
    pickle.dump(in_dict, pickle_file)
    pickle_file.close()

def make_anomalies_database(daily_file_name="daily_rad_flex_coffeevJune_cn.csv",estado='MG',
                           save_path="C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/"):
    df_main=pd.read_csv(save_path+daily_file_name)
    df_main['mm_dd']=(df_main['date']%10000).astype(int)
    df_main['year']=(df_main['date']//10000)
    df_main=df_main[df_main['mm_dd']!=229]
    df_main=df_main.sort_values(by='date').reset_index(drop=True)
    #groups are 0101 to 0107 as grp1, 0102 to 0108 as grp2 and so on...
    #print(df_main.shape)

    mm_dds=df_main['mm_dd'].unique()
    dates_arr=df_main['date'].values #len=13140
    values_arr=df_main[estado].values
    z_score,high,low={},{},{}
    for i in range(365):

        rolled_dates=np.roll(dates_arr,-i)
        rolled_values=np.roll(values_arr,-i)
        ix=np.arange(7)[:, None] + np.arange(0, 13140, 365)[None, :] ## 365*36 = 13140

        dates=np.take(rolled_dates,ix)
        values=np.take(rolled_values,ix)

        values_mean,values_std=values.mean(),values.std()
        values_zscore=(values-values_mean)/values_std ### Z-SCORES

        low_qtl,high_qtl=np.quantile(values,0.05),np.quantile(values,0.95)
        values_low,values_high=values<=low_qtl,values>=high_qtl

        if i<=361:
            z_score[mm_dds[i+3]],high[mm_dds[i+3]],low[mm_dds[i+3]]={},{},{}
            z_score[mm_dds[i+3]]['date']=dates.flatten()
            z_score[mm_dds[i+3]]['value']=values.flatten()
            z_score[mm_dds[i+3]]['z-scores']=values_zscore.flatten()
            high[mm_dds[i+3]]['date']=dates.flatten()
            high[mm_dds[i+3]]['value']=values.flatten()
            high[mm_dds[i+3]]['high']=values_high.flatten()
            low[mm_dds[i+3]]['date']=dates.flatten()
            low[mm_dds[i+3]]['value']=values.flatten()
            low[mm_dds[i+3]]['low']=values_low.flatten()
        else:
            z_score[mm_dds[i-362]],high[mm_dds[i-362]],low[mm_dds[i-362]]={},{},{}
            z_score[mm_dds[i-362]]['date']=dates.flatten()
            z_score[mm_dds[i-362]]['value']=values.flatten()
            z_score[mm_dds[i-362]]['z-scores']=values_zscore.flatten()
            high[mm_dds[i-362]]['date']=dates.flatten()
            high[mm_dds[i-362]]['value']=values.flatten()
            high[mm_dds[i-362]]['high']=values_high.flatten()
            low[mm_dds[i-362]]['date']=dates.flatten()
            low[mm_dds[i-362]]['value']=values.flatten()
            low[mm_dds[i-362]]['low']=values_low.flatten()

        variable_string=daily_file_name[6:-19]
        dict_name=variable_string+"_"+estado
        
    save_dictionaries_as_pickle(in_dict=z_score,save_path=save_path,dict_name="z_score"+dict_name)
    save_dictionaries_as_pickle(in_dict=high,save_path=save_path,dict_name="high"+dict_name)
    save_dictionaries_as_pickle(in_dict=low,save_path=save_path,dict_name="low"+dict_name)

    #return z_score

In [47]:
save_path="C://Users/nb/Desktop/brazil_climate_data/daily_estados_climate/"
print('The files available for compilatation are: ')
all_files=os.listdir(save_path)
print(all_files)
cn_files=[i for i in all_files if '_cn.csv' in i]
print("* ============================ *")
print("Using the files with _cn suffix because they have the full cycle of observations")
print(cn_files)

The files available for compilatation are: 
['daily_pet_coffeevJune.csv', 'daily_pet_coffeevJune_cn.csv', 'daily_prec_flex_coffeevJune.csv', 'daily_prec_flex_coffeevJune_cn.csv', 'daily_rad_flex_coffeevJune.csv', 'daily_rad_flex_coffeevJune_cn.csv', 'daily_vpdmax_coffeevJune.csv', 'daily_vpdmax_coffeevJune_cn.csv', 'highpet_BA', 'highpet_ES', 'highpet_MG', 'highpet_PR', 'highpet_SP', 'highprec_flex_BA', 'highprec_flex_ES', 'highprec_flex_MG', 'highprec_flex_PR', 'highprec_flex_SP', 'highrad_flex_BA', 'highrad_flex_ES', 'highrad_flex_MG', 'highrad_flex_PR', 'highrad_flex_SP', 'highvpdmax_BA', 'highvpdmax_ES', 'highvpdmax_MG', 'highvpdmax_PR', 'highvpdmax_SP', 'lowpet_BA', 'lowpet_ES', 'lowpet_MG', 'lowpet_PR', 'lowpet_SP', 'lowprec_flex_BA', 'lowprec_flex_ES', 'lowprec_flex_MG', 'lowprec_flex_PR', 'lowprec_flex_SP', 'lowrad_flex_BA', 'lowrad_flex_ES', 'lowrad_flex_MG', 'lowrad_flex_PR', 'lowrad_flex_SP', 'lowvpdmax_BA', 'lowvpdmax_ES', 'lowvpdmax_MG', 'lowvpdmax_PR', 'lowvpdmax_SP', 'z_

In [46]:
for file_name in cn_files:
    for estado in estados:
        make_anomalies_database(save_path=save_path,daily_file_name=file_name,estado=estado)

## COLOMBIA

In [ ]:
import cdsapi
years= [2000, 2001, 2002, 2003, 2004, 2005,
        2006, 2007, 2008, 2009, 2010, 2011,
        2012, 2013, 2014, 2015, 2016, 2017,
        2018, 2019, 2020, 2021, 2022, 2023,
        2024, 2025]
dataset = "sis-agrometeorological-indicators"

In [ ]:
var_name = "vapour_pressure_deficit_at_maximum_temperature"

for year in years:
    request = {
        "variable": var_name,
        "year": [year],
        "month": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12"
        ],
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "version": "2_0",
        "area": [15, -80, -5, -65]
    }

    client = cdsapi.Client()
    client.retrieve(dataset, request).download(f"C://Users/nb/Desktop/colombia_climate_data/{var_name}_{year}.zip")

In [ ]:
var_name = "precipitation_flux"

for year in years:
    request = {
        "variable": var_name,
        "year": [year],
        "month": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12"
        ],
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "version": "2_0",
        "area": [15, -80, -5, -65]
    }

    client = cdsapi.Client()
    client.retrieve(dataset, request).download(f"C://Users/nb/Desktop/colombia_climate_data/{var_name}_{year}.zip")

In [ ]:
var_name ="solar_radiation_flux"

for year in years:
    request = {
        "variable": var_name,
        "year": [year],
        "month": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12"
        ],
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "version": "2_0",
        "area": [15, -80, -5, -65]
    }

    client = cdsapi.Client()
    client.retrieve(dataset, request).download(f"C://Users/nb/Desktop/colombia_climate_data/{var_name}_{year}.zip")

In [ ]:
var_name ="reference_evapotranspiration"

for year in years:
    request = {
        "variable": var_name,
        "year": [year],
        "month": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12"
        ],
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "version": "2_0",
        "area": [15, -80, -5, -65]
    }

    client = cdsapi.Client()
    client.retrieve(dataset, request).download(f"C://Users/nb/Desktop/colombia_climate_data/{var_name}_{year}.zip")

In [ ]:
raw_files=['precipitation_flux_raw','solar_radiation_flux_raw',
           'vapour_pressure_deficit_at_max_temp_raw','reference_evapotranspiration_raw'] #

years= [2000, 2001, 2002, 2003, 2004, 2005,
        2006, 2007, 2008, 2009, 2010, 2011,
        2012, 2013, 2014, 2015, 2016, 2017,
        2018, 2019, 2020, 2021, 2022, 2023,
        2024, 2025, 1990,1991,1992,1993,1994,1995,
          1996,1997,1998,1999]
for raw in raw_files:
    file_txt=raw.replace('_raw','')
    raw_zip_folder = f'C://Users/nb/Desktop/colombia_climate_data/{raw}/'
    extract_folder = f'C://Users/nb/Desktop/colombia_climate_data/{file_txt}/'
    
    output_file = f'combined_{file_txt}_1990_1999.nc'

    os.makedirs(raw_zip_folder, exist_ok=True)

    print("Extracting files...")
    zip_files = glob.glob(os.path.join(raw_zip_folder, "*.zip"))

    for zip_path in zip_files:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            nc_files = [f for f in zip_ref.namelist() if f.endswith('.nc')]
            zip_ref.extractall(path=extract_folder, members=nc_files)
        print(f"Done: {os.path.basename(zip_path)}")


In [ ]:
pname="C://Users/nb/Desktop/colombia_climate_data/precipitation_flux/"
prec_flux_xr = xr.open_mfdataset(pname+'*.nc',preprocess=make_climate_xr)

In [ ]:
prec_flux_xr.to_netcdf("C://Users/nb/Desktop/colombia_climate_data/_climate_data/coffee_prec_flux.nc")

In [ ]:
pname="C://Users/nb/Desktop/colombia_climate_data/solar_radiation_flux/"
rad_flux_xr = xr.open_mfdataset(pname+'*.nc',preprocess=make_climate_xr)

In [ ]:
rad_flux_xr.to_netcdf("C://Users/nb/Desktop/colombia_climate_data/coffee_rad_flux.nc")

In [ ]:
pname="C://Users/nb/Desktop/colombia_climate_data/reference_evapotranspiration/"
pet_xr = xr.open_mfdataset(pname+'*.nc',preprocess=make_climate_xr)

In [ ]:
pet_xr.to_netcdf("C://Users/nb/Desktop/colombia_climate_data/coffee_pet.nc")

In [ ]:
pname="C://Users/nb/Desktop/colombia_climate_data/vapour_pressure_deficit_at_max_temp/"
vpd_max_xr =xr.open_mfdataset(pname+'*.nc',preprocess=make_climate_xr)

In [ ]:
vpd_max_xr.to_netcdf("C://Users/nb/Desktop/colombia_climate_data/coffee_vpd_max.nc")